# DATA ANALYSIS 
## DATA COVERAGE 

### 2. COMBINING DATA SETS

In [35]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm


# ============================================================
# 1. File paths
# ============================================================

file_paths = {
    'imf': 'IMF_final_variant_country_availability.csv',
    'unsd': 'UNSD_Environmental_Indicators_Metadata_Availability.xlsx',
    'wb': 'WB_final_CWON_full_indicator_variations.csv'
}


# ============================================================
# 2. Settings
# ============================================================

meta_cols = [
    'no',
    'indicator',
    'Definition',
    'unit',
    'source'
]

source_prefix = {
    'wb': 'wb',
    'unsd': 'un',
    'imf': 'imf'
}

ordered_sources = [
    'wb',
    'unsd',
    'imf'
]


# ============================================================
# 3. Load datasets
# ============================================================

loaded = {}

for source_name, path_val in tqdm(file_paths.items(), desc='Loading files'):
    if path_val.endswith('.csv'):
        loaded[source_name] = pd.read_csv(
            path_val,
            encoding='utf-8'
        )

    elif path_val.endswith('.xlsx'):
        loaded[source_name] = pd.read_excel(path_val)

    else:
        raise ValueError('Unsupported file type: ' + path_val)


# ============================================================
# 4. Inspect loaded data
# ============================================================

for source_name, df_val in loaded.items():
    print('\nDataset:')
    print(source_name)

    print('Shape:')
    print(df_val.shape)

    print('Preview:')
    display(df_val.head())


# ============================================================
# 5. Identify country columns
# ============================================================

country_cols_by_source = {}

for source_name, df_val in loaded.items():
    country_cols_by_source[source_name] = [
        col_val for col_val in df_val.columns
        if col_val not in meta_cols
    ]

base_country_cols = country_cols_by_source['imf']


# ============================================================
# 6. Check whether country columns match across datasets
# ============================================================

alignment_rows = []

for source_name, country_cols_val in country_cols_by_source.items():
    alignment_rows.append({
        'dataset': source_name,
        'rows': len(loaded[source_name]),
        'total_columns': len(loaded[source_name].columns),
        'country_columns': len(country_cols_val),
        'same_country_order_as_imf': country_cols_val == base_country_cols,
        'missing_vs_imf': len(set(base_country_cols) - set(country_cols_val)),
        'extra_vs_imf': len(set(country_cols_val) - set(base_country_cols))
    })

alignment_df = pd.DataFrame(alignment_rows)

print('\nCountry column alignment check:')
display(alignment_df)


# ------------------------------------------------------------
# 6A. Stop if country columns do not match
# ------------------------------------------------------------

if not alignment_df['same_country_order_as_imf'].all():
    raise ValueError(
        'Country columns do not match exactly across datasets. '
        'Please inspect alignment_df before combining.'
    )

country_cols = base_country_cols


# ============================================================
# 7. Prefix original IDs and combine datasets vertically
# ============================================================

combined_parts = []

for source_name in tqdm(ordered_sources, desc='Combining datasets'):
    df_part = loaded[source_name].copy()

    prefix_val = source_prefix[source_name]

    # Store prefixed source-specific ID in the original column
    df_part['original'] = [
        prefix_val + '_' + str(row_idx + 1).zfill(4)
        for row_idx in range(len(df_part))
    ]

    # Add source dataset label
    df_part['dataset'] = source_name

    # Reorder columns temporarily.
    # The final no column will be created after combining all datasets.
    df_part = df_part[
        [
            'indicator',
            'Definition',
            'unit',
            'source',
            'original',
            'dataset'
        ]
        + country_cols
    ]

    combined_parts.append(df_part)

combined_wide_df = pd.concat(
    combined_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# 7A. Add final sequential no column
# ------------------------------------------------------------

combined_wide_df.insert(
    0,
    'no',
    range(1, len(combined_wide_df) + 1)
)


# ============================================================
# 8. Validate combined wide dataset
# ============================================================

validation_summary = pd.DataFrame({
    'check': [
        'total_rows',
        'total_columns',
        'country_columns',
        'unique_no_values',
        'duplicate_no_values',
        'unique_original_values',
        'duplicate_original_values'
    ],
    'value': [
        len(combined_wide_df),
        len(combined_wide_df.columns),
        len(country_cols),
        combined_wide_df['no'].nunique(),
        combined_wide_df['no'].duplicated().sum(),
        combined_wide_df['original'].nunique(),
        combined_wide_df['original'].duplicated().sum()
    ]
})

source_summary = (
    combined_wide_df
    .groupby('dataset')
    .agg(
        rows=('no', 'count'),
        first_no=('no', 'min'),
        last_no=('no', 'max'),
        unique_original=('original', 'nunique')
    )
    .reset_index()
)

print('\nValidation summary:')
display(validation_summary)

print('\nSource summary:')
display(source_summary)

print('\nCombined wide preview:')
display(combined_wide_df.head())


# ============================================================
# 9. Create availability summary
# ============================================================

availability_summary_rows = []

for dataset_name, df_group in tqdm(
    combined_wide_df.groupby('dataset'),
    desc='Summarizing availability'
):
    country_values = df_group[country_cols]

    available_cells = int((country_values == 1).sum().sum())
    total_country_cells = int(country_values.size)

    availability_summary_rows.append({
        'dataset': dataset_name,
        'indicators': len(df_group),
        'country_columns': len(country_cols),
        'available_cells': available_cells,
        'total_country_cells': total_country_cells,
        'availability_rate': available_cells / total_country_cells
    })

availability_summary_df = pd.DataFrame(availability_summary_rows)

print('\nAvailability summary:')
display(availability_summary_df)


# ============================================================
# 10. Save outputs
# ============================================================

wide_file = 'combined_country_availability_wide.csv'
excel_file = 'combined_country_availability.xlsx'


# ------------------------------------------------------------
# 10A. Save CSV output
# ------------------------------------------------------------

combined_wide_df.to_csv(
    wide_file,
    index=False,
    encoding='utf-8-sig'
)


# ------------------------------------------------------------
# 10B. Save Excel output
# ------------------------------------------------------------

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    combined_wide_df.to_excel(
        writer,
        sheet_name='wide_country_columns',
        index=False
    )

    validation_summary.to_excel(
        writer,
        sheet_name='validation_summary',
        index=False
    )

    source_summary.to_excel(
        writer,
        sheet_name='source_summary',
        index=False
    )

    alignment_df.to_excel(
        writer,
        sheet_name='country_alignment',
        index=False
    )

    availability_summary_df.to_excel(
        writer,
        sheet_name='availability_summary',
        index=False
    )


# ------------------------------------------------------------
# 10C. Confirm saved files
# ------------------------------------------------------------

print('\nSaved files:')
print(wide_file)
print(excel_file)

Loading files:   0%|          | 0/3 [00:00<?, ?it/s]


Dataset:
imf
Shape:
(1089, 223)
Preview:


,no,indicator,Definition,unit,source,Afghanistan,Albania,Algeria,American Samoa,Andorra,...,Uruguay,Uzbekistan,Vanuatu,Venezuela,Viet Nam,Virgin Islands (U.S.),West Bank and Gaza,Yemen,Zambia,Zimbabwe
0,03-0001,IMF estimated CO2 emissions under a Business a...,"Environment, Climate Change, Greenhouse Gas Em...",Units,United Nations Framework Convention on Climate...,1,1,1,0,0,...,1,1,1,1,1,0,0,1,1,1
1,03-0001,IMF estimated CO2 emissions under a Business a...,"Environment, Climate Change, Greenhouse Gas Em...",Units,United Nations Framework Convention on Climate...,1,1,1,0,0,...,1,1,1,1,1,0,0,1,1,1
2,03-0002,IMF estimated GHG emissions excluding land-use...,"Environment, Climate Change, Greenhouse Gas Em...",Units,United Nations Framework Convention on Climate...,1,1,1,0,0,...,1,1,1,1,1,0,0,1,1,1
3,03-0002,IMF estimated GHG emissions excluding land-use...,"Environment, Climate Change, Greenhouse Gas Em...",Units,United Nations Framework Convention on Climate...,1,1,1,0,0,...,1,1,1,1,1,0,0,1,1,1
4,03-0003,IMF estimated GHG emissions including land-use...,"Environment, Climate Change, Greenhouse Gas Em...",Units,United Nations Framework Convention on Climate...,1,1,1,0,0,...,1,1,1,1,1,0,0,1,1,1



Dataset:
unsd
Shape:
(158, 223)
Preview:


,no,indicator,Definition,unit,source,Afghanistan,Albania,Algeria,American Samoa,Andorra,...,Uruguay,Uzbekistan,Vanuatu,Venezuela,Viet Nam,Virgin Islands (U.S.),West Bank and Gaza,Yemen,Zambia,Zimbabwe
0,1,Actual evapotranspiration,Actual evapotranspiration: Total actual volume...,Million cubic metres,https://unstats.un.org/unsd/envstats/Questionn...,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2,% change of agricultural area since 1990,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,1,1,1,...,1,0,1,1,1,0,0,1,1,1
2,3,% of total land area covered by agricultural a...,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,1,1,1,...,1,1,1,1,1,0,1,1,1,1
3,4,Agricultural area actually irrigated in 2021,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,5,Agricultural area in 2021,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,1,1,1,...,1,1,1,1,1,0,1,1,1,1


...

,no,indicator,Definition,unit,source,original,dataset,Afghanistan,Albania,Algeria,...,Uruguay,Uzbekistan,Vanuatu,Venezuela,Viet Nam,Virgin Islands (U.S.),West Bank and Gaza,Yemen,Zambia,Zimbabwe
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,0,1,0,...,1,0,0,0,1,0,0,0,1,1
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,0,1,0,...,1,0,0,0,1,0,0,0,1,1
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,0,1,0,...,1,0,0,0,1,0,0,0,1,1
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,0,1,0,...,1,0,0,0,1,0,0,0,1,1
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,0,1,0,...,1,0,0,0,1,0,0,0,1,1


Summarizing availability:   0%|          | 0/3 [00:00<?, ?it/s]


Availability summary:


,dataset,indicators,country_columns,available_cells,total_country_cells,availability_rate
0,imf,1089,218,114441,237402,0.482056
1,unsd,158,218,15581,34444,0.452357
2,wb,76,218,7974,16568,0.481289



Saved files:
combined_country_availability_wide.csv
combined_country_availability.xlsx
